In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from matplotlib import pyplot
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
#nltk.download('punkt_tab')
#nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk import word_tokenize
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report
import os
import wandb
import tensorflow as tf
import copy
wandb.login()
#from wandb.integration.keras import WandbCallback


I0000 00:00:1776092635.340945  856700 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: linneahejsupergroup (linneahejsupergroup-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
torch.cuda.empty_cache()
import gc
gc.collect()
print(torch.cuda.memory_allocated() / 1024**2, "MB allocated")  # should be ~0
print(torch.cuda.memory_reserved() / 1024**2, "MB reserved")

0.0 MB allocated
0.0 MB reserved


In [3]:
def preprocess_pandas(data, columns):
    df_ = pd.DataFrame(columns=columns)
    data['Sentence'] = data['Sentence'].str.lower()
    data['Sentence'] = data['Sentence'].replace('[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)                      # remove emails
    data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
    data['Sentence'] = data['Sentence'].str.replace('[^\w\s]','')                                                       # remove special characters
    data['Sentence'] = data['Sentence'].replace('\d', '', regex=True)                                                   # remove numbers
    for index, row in data.iterrows():
        word_tokens = word_tokenize(row['Sentence'])
        filtered_sent = [w for w in word_tokens if not w in stopwords.words('english')]
        df_.loc[len(df_)] = {
            "index": row['index'],
            "Class": row['Class'],
            "Sentence": " ".join(filtered_sent)
        }
    return data

In [12]:
def get_loaders(file_path):
    data = pd.read_csv(file_path, delimiter='\t', header=None)
    data.columns = ['Sentence', 'Class']
    data['index'] = data.index                                          # add new column index
    columns = ['index', 'Class', 'Sentence']
    data = preprocess_pandas(data, columns)                             # pre-process
    training_data, validation_data, training_labels, validation_labels = train_test_split( # split the data into training, validation, and test splits
        data['Sentence'].values.astype('U'),
        data['Class'].values.astype('int32'),
        test_size=0.10,
        random_state=0,
        shuffle=True
    )

    # vectorize data using TFIDF and transform for PyTorch for scalability
    word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=50000, max_df=0.5, use_idf=True, norm='l2')
    training_data = word_vectorizer.fit_transform(training_data)        # transform texts to sparse matrix
    training_data = training_data.todense()                             # convert to dense matrix for Pytorch
    vocab_size = len(word_vectorizer.vocabulary_)
    validation_data = word_vectorizer.transform(validation_data)
    validation_data = validation_data.todense()
    train_x_tensor = torch.from_numpy(np.array(training_data)).type(torch.FloatTensor)
    train_y_tensor = torch.from_numpy(np.array(training_labels)).long()
    validation_x_tensor = torch.from_numpy(np.array(validation_data)).type(torch.FloatTensor)
    validation_y_tensor = torch.from_numpy(np.array(validation_labels)).long()
    print("Data loading complete")
    return train_x_tensor, train_y_tensor, validation_x_tensor, validation_y_tensor, vocab_size


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

if __name__ == "__main__":
    # get data, pre-process and split
    data_a = pd.read_csv("amazon_cells_labelled.txt", delimiter='\t', header=None)
    data_a.columns = ['Sentence', 'Class']
    data_a['index'] = data_a.index                                          # add new column index
    columns_a = ['index', 'Class', 'Sentence']
    data_a = preprocess_pandas(data_a, columns_a)                             # pre-process
    training_data_a, validation_data_a, training_labels_a, validation_labels_a = train_test_split( # split the data into training, validation, and test splits
        data_a['Sentence'].values.astype('U'),
        data_a['Class'].values.astype('int32'),
        test_size=0.15,
        random_state=0,
        shuffle=True
    )

tokenizer = Tokenizer()
tokenizer.fit_on_texts(training_data_a)          # only training data

sequences_train = tokenizer.texts_to_sequences(training_data_a)
sequences_val   = tokenizer.texts_to_sequences(validation_data_a)

vocab_size = len(tokenizer.word_index) + 1       # e.g. 1500 unique words → vocab_size = 1501

maxlen = max(len(seq) for seq in sequences_train) + 10

sequences_pad_train = pad_sequences(sequences_train, maxlen=maxlen)
sequences_pad_val   = pad_sequences(sequences_val,   maxlen=maxlen)

train_x = torch.from_numpy(np.array(sequences_pad_train)).type(torch.FloatTensor)
train_y = torch.from_numpy(np.array(training_labels_a)).long()
test_x  = torch.from_numpy(np.array(sequences_pad_val)).type(torch.FloatTensor)
test_y  = torch.from_numpy(np.array(validation_labels_a)).long()

print(f"Vocab size: {vocab_size}, Max sequence length: {maxlen}")

Vocab size: 1663, Max sequence length: 39


Task 1.1

In [5]:
class SimpleNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, output_size, dropout=0.5):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(vocab_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, (hidden_size))
        self.fc3 = nn.Linear((hidden_size), (hidden_size))
        self.fc4 = nn.Linear((hidden_size), output_size)
        self.dropout = nn.Dropout(dropout)


    def forward(self, x):
        x = F.leaky_relu(self.fc1(x))
        x = self.dropout(x)
        x = F.leaky_relu(self.fc2(x))
        x = self.dropout(x)
        x = F.leaky_relu(self.fc3(x))
        x = self.dropout(x)
        x = self.fc4(x)
        return x

In [8]:
'''class LSTM_model(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_size, num_layers=4, dropout=0.5):
        super(LSTM_model, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x.long()
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x)
        lstm_out = lstm_out[:, -1, :]
        lstm_out = self.dropout(lstm_out)
        out = self.fc(lstm_out)
        return out'''
class LSTM_model(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_size, num_layers=2, dropout=0.5, bidirectional=True):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional = bidirectional
        )
        num_directions = 2 if self.bidirectional else 1
        self.fc = nn.Linear(hidden_dim * num_directions, output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x.long()
        x = self.dropout(self.embedding(x))
        x, (h,c) = self.lstm(x)
        x = x[:, -1, :]
        x = self.dropout(x)
        
        x = self.fc(x)
        return x
'''
    def forward(self, x):
        #x = x.long()
        #x = self.dropout(self.embedding(x))

        _, (hidden, _) = self.lstm(x)
        out = hidden[-1]

        out = self.dropout(out)
        out = self.fc(out)
        return out'''


    

'\n    def forward(self, x):\n        #x = x.long()\n        #x = self.dropout(self.embedding(x))\n\n        _, (hidden, _) = self.lstm(x)\n        out = hidden[-1]\n\n        out = self.dropout(out)\n        out = self.fc(out)\n        return out'

Training functions

In [4]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        running_loss += criterion(outputs, labels).item()
        correct += outputs.argmax(1).eq(labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, 100 * correct / total

def train(model, train_loader, val_loader, criterion, optimizer, device, tag, num_epochs=10, early_stopping_patience=3, scheduler=None):
    model.to(device)
    best_val_loss = 1000000000000
    epochs_without_improvement = 0
    train_losses, val_losses, val_accuracies = [], [], []
    best_model = None
    for epoch in range(num_epochs):
        total = 0
        correct = 0
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            optimizer.zero_grad()
            
            running_loss += loss.item()
            correct += outputs.argmax(1).eq(labels).sum().item()
            total += labels.size(0)
        train_loss = running_loss / total
        train_accuracy = 100 * correct / total
        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)
        if scheduler:
            scheduler.step(val_loss)#remove val_loss depending on if the scheduler needs it or not
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            best_model = copy.deepcopy(model)
            torch.save(best_model.state_dict(), f'{tag}_best_model.pth') #saves the best model dict to file in case of crashes or memory issues
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= early_stopping_patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)
        wandb.log({"Train Loss": train_loss, "Train Accuracy": train_accuracy, "Val Loss": val_loss, "Val Accuracy": val_accuracy, "epoch": epoch+1, "LR": scheduler.get_last_lr()})
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%')
    return train_losses, val_losses, val_accuracies, best_model

NameError: name 'torch' is not defined

In [14]:
LR = 0.0001
BATCH_SIZE = 16
NUM_EPOCHS = 10000
EARLY_STOP = 300

In [15]:
file_path = "/lab1/amazon_cells_labelled.txt"

#train_x, train_y, validation_x, validation_y, vocab_size = get_loaders(file_path)
device='cuda' if torch.cuda.is_available() else 'cpu'
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(test_x, test_y), batch_size=BATCH_SIZE, shuffle=False)

In [48]:



NNmodel = SimpleNN(vocab_size=vocab_size, hidden_size=16, output_size=2)


criterion = nn.CrossEntropyLoss()
optimizer_NN = optim.AdamW(NNmodel.parameters(), lr=LR)

scheduler_NN = optim.lr_scheduler.CosineAnnealingLR(optimizer_NN, T_max=NUM_EPOCHS)




wandb.init(project="Lab1", name="NN_model", config={
    "learning_rate": LR,
    "batch_size": BATCH_SIZE,
    "epochs": NUM_EPOCHS,
    "early_stopping": EARLY_STOP,
    "dropout": 0.5
})
tf.keras.backend.clear_session()
torch.cuda.empty_cache()
train(NNmodel, train_loader, val_loader, criterion, optimizer_NN, device, tag="NN_model", num_epochs=NUM_EPOCHS, early_stopping_patience=EARLY_STOP, scheduler=scheduler_NN)
wandb.finish()


Data loading complete


Epoch [1/10000], Train Loss: 0.1770, Val Loss: 0.1738, Val Accuracy: 53.00%
Epoch [2/10000], Train Loss: 0.1769, Val Loss: 0.1734, Val Accuracy: 53.00%
Epoch [3/10000], Train Loss: 0.1754, Val Loss: 0.1730, Val Accuracy: 53.00%
Epoch [4/10000], Train Loss: 0.1743, Val Loss: 0.1727, Val Accuracy: 53.00%
Epoch [5/10000], Train Loss: 0.1741, Val Loss: 0.1724, Val Accuracy: 53.00%
Epoch [6/10000], Train Loss: 0.1740, Val Loss: 0.1720, Val Accuracy: 53.00%
Epoch [7/10000], Train Loss: 0.1735, Val Loss: 0.1715, Val Accuracy: 53.00%
Epoch [8/10000], Train Loss: 0.1724, Val Loss: 0.1708, Val Accuracy: 53.00%
Epoch [9/10000], Train Loss: 0.1705, Val Loss: 0.1696, Val Accuracy: 57.00%
Epoch [10/10000], Train Loss: 0.1680, Val Loss: 0.1679, Val Accuracy: 72.00%
Epoch [11/10000], Train Loss: 0.1652, Val Loss: 0.1657, Val Accuracy: 84.00%
Epoch [12/10000], Train Loss: 0.1604, Val Loss: 0.1626, Val Accuracy: 84.00%
Epoch [13/10000], Train Loss: 0.1561, Val Loss: 0.1588, Val Accuracy: 84.00%
Epoch [1

Train Accuracy,▁▇▇▇████████████████████████████████████
Train Loss,█▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Val Accuracy,▁▆▅▅▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█
Val Loss,▂▂▂▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇█████
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
Train Accuracy,100
Train Loss,0.00092
Val Accuracy,85
Val Loss,0.81108
epoch,324


In [16]:
LR = 2e-3
BATCH_SIZE = 16
NUM_EPOCHS = 10000
EARLY_STOP = 300

In [17]:
LSTMmodel = LSTM_model(vocab_size=vocab_size, embedding_dim=128, hidden_dim=128, output_size=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer_LSTM = optim.AdamW(
    LSTMmodel.parameters(),
    lr= LR,   
    weight_decay=1e-2
)
scheduler_LSTM = scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_LSTM,
    mode='min',        
    factor=0.5,        
    patience=20,        
    threshold=1e-3,    
    min_lr=1e-6
)
wandb.init(project="Lab1", name="LSTM_model", config={
    "learning_rate": LR,
    "batch_size": BATCH_SIZE,
    "epochs": NUM_EPOCHS,
    "early_stopping": EARLY_STOP,
    "dropout": 0.5
})
tf.keras.backend.clear_session()
torch.cuda.empty_cache()
train(LSTMmodel, train_loader, val_loader, criterion, optimizer_LSTM, device, tag="LSTM_model", num_epochs=NUM_EPOCHS, early_stopping_patience=EARLY_STOP, scheduler=scheduler_LSTM)

wandb.finish()

Epoch [1/10000], Train Loss: 0.0423, Val Loss: 0.0446, Val Accuracy: 54.00%
Epoch [2/10000], Train Loss: 0.0368, Val Loss: 0.0407, Val Accuracy: 66.67%
Epoch [3/10000], Train Loss: 0.0311, Val Loss: 0.0410, Val Accuracy: 73.33%
Epoch [4/10000], Train Loss: 0.0237, Val Loss: 0.0404, Val Accuracy: 76.00%
Epoch [5/10000], Train Loss: 0.0197, Val Loss: 0.0406, Val Accuracy: 74.67%
Epoch [6/10000], Train Loss: 0.0171, Val Loss: 0.0401, Val Accuracy: 80.00%
Epoch [7/10000], Train Loss: 0.0146, Val Loss: 0.0495, Val Accuracy: 74.67%
Epoch [8/10000], Train Loss: 0.0109, Val Loss: 0.0605, Val Accuracy: 75.33%
Epoch [9/10000], Train Loss: 0.0086, Val Loss: 0.0589, Val Accuracy: 80.00%
Epoch [10/10000], Train Loss: 0.0084, Val Loss: 0.0590, Val Accuracy: 79.33%
Epoch [11/10000], Train Loss: 0.0085, Val Loss: 0.0615, Val Accuracy: 80.00%
Epoch [12/10000], Train Loss: 0.0092, Val Loss: 0.0690, Val Accuracy: 77.33%
Epoch [13/10000], Train Loss: 0.0068, Val Loss: 0.0689, Val Accuracy: 77.33%
Epoch [1

Train Accuracy,▁▅▆▇████████████████████████████████████
Train Loss,▃▃▄█▇▄▂▁▃▃▃▂▃▂▂▁▃▁▂▂▂▁▁▂▂▁▁▁▃▁▄▂▂▁▂▁▂▁▁▁
Val Accuracy,▁▆▆▇█▇██▇█▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
Val Loss,▁▁▂▄▅▅▆▆▇▇█▇████████████████████████████
epoch,▁▁▁▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
Train Accuracy,99.88235
Train Loss,8e-05
Val Accuracy,78
Val Loss,0.12846
epoch,305


In [18]:
print(f"Vocab size: {vocab_size}")
print(torch.cuda.memory_allocated() / 1024**2, "MB allocated")
print(torch.cuda.memory_reserved() / 1024**2, "MB reserved")

Vocab size: 1663
29.85791015625 MB allocated
130.0 MB reserved


In [27]:
wandb.finish()

In [ ]:
import kagglehub
import pandas as pd
import numpy as np

if __name__ == "__main__":
    path = kagglehub.dataset_download("rogate16/amazon-reviews-2018-full-dataset")

    sentence_chunks = []
    label_chunks = []

    for chunk in pd.read_csv(
        path + "/amazon_reviews.csv",
        chunksize=50000,
        usecols=['description', 'rating']
    ):
        chunk = chunk.dropna(subset=['description', 'rating'])
        sentence_chunks.append(chunk['description'].values.astype('U'))
        label_chunks.append(chunk['rating'].values.astype('int32'))
        print(f"Chunks loaded: {len(sentence_chunks)}, samples so far: {sum(len(c) for c in sentence_chunks)}")

    sentences = np.concatenate(sentence_chunks)
    labels    = np.concatenate(label_chunks)

    # Clean list-like description strings e.g. ["Some text..."]
    import ast
    def clean_description(text):
        try:
            parsed = ast.literal_eval(text)
            return ' '.join(parsed) if isinstance(parsed, list) else str(parsed)
        except:
            return str(text)

    sentences = np.array([clean_description(s) for s in sentences])

    training_data_a, validation_data_a, training_labels_a, validation_labels_a = train_test_split(
        sentences,
        labels,
        test_size=0.15,
        random_state=0,
        shuffle=True
    )

tokenizer = Tokenizer()
tokenizer.fit_on_texts(training_data_a)
sequences_train = tokenizer.texts_to_sequences(training_data_a)
sequences_val   = tokenizer.texts_to_sequences(validation_data_a)
vocab_size = len(tokenizer.word_index) + 1
maxlen = max(len(seq) for seq in sequences_train) + 10
sequences_pad_train = pad_sequences(sequences_train, maxlen=maxlen)
sequences_pad_val   = pad_sequences(sequences_val,   maxlen=maxlen)
train_x = torch.from_numpy(np.array(sequences_pad_train)).type(torch.FloatTensor)
train_y = torch.from_numpy(np.array(training_labels_a)).long()
test_x  = torch.from_numpy(np.array(sequences_pad_val)).type(torch.FloatTensor)
test_y  = torch.from_numpy(np.array(validation_labels_a)).long()
print(f"Vocab size: {vocab_size}, Max sequence length: {maxlen}")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunks loaded: 1, samples so far: 50000
Chunks loaded: 2, samples so far: 99998
Chunks loaded: 3, samples so far: 149993
Chunks loaded: 4, samples so far: 199993
Chunks loaded: 5, samples so far: 249993
